In [1]:
suppressPackageStartupMessages({
  library(data.table)  # fread, rbindlist
  library(dplyr)       # filter, mutate, arrange, distinct
  library(tidyr)       # pivot_wider
  library(stringr)     # str_to_title, gsub wrappers
  library(ggplot2)     # plotting
  library(ggrepel)     # geom_text_repel
  library(grid)        # unit()
  library(DESeq2)      # DE analysis
  library(qvalue)      # qvalue FDR
  library(parallel)    # mclapply
  library(tidyverse)
})

source('../../00-utilities/functions/r/base-deseq2.R')

In [2]:
metadata = read.csv("../../../data/rna/pseudobulk/outputs/bmmc_sample_kit_metadata.csv")

In [3]:
# Define comparisons and gene list files - Healthy vs disease comparisons
comparisons <- list(
  list(tp1 = "PreTx",   tp2 = "Healthy", gene_file = "bmmc_l3_healthy-vs-PreTx_filtered_gene_list.csv"),
  list(tp1 = "EI",      tp2 = "Healthy", gene_file = "bmmc_l3_healthy-vs-EI_filtered_gene_list.csv"),
  list(tp1 = "ASCT90d", tp2 = "Healthy", gene_file = "bmmc_l3_healthy-vs-ASCT90d_filtered_gene_list.csv"),
  list(tp1 = "ASCT1y",  tp2 = "Healthy", gene_file = "bmmc_l3_healthy-vs-ASCT1y_filtered_gene_list.csv"),
  list(tp1 = "ASCT2y",  tp2 = "Healthy", gene_file = "bmmc_l3_healthy-vs-ASCT2y_filtered_gene_list.csv")
)

In [4]:
# Loop over each comparison
for (comp in comparisons) {
  timepoint_1 <- comp$tp1
  timepoint_2 <- comp$tp2
  gene_file <- comp$gene_file

  # Define output file using timepoints
  out_file <- paste0("deseq2_results_bmmc_", timepoint_1, "_vs_", timepoint_2, ".csv")

  # Filter metadata for only the two timepoints
  metadata_sub <- metadata %>%
    filter(
      label.visitDetails %in% c(timepoint_1, timepoint_2)
    )

  # List of pseudobulk expression files
  aggregated_count_file_list <- paste0(
    "../../../data/rna/pseudobulk/outputs/bmmc_l3_raw_gexp_celltypes_per_samplekit/",
    unique(metadata_sub$sample.sampleKitGuid), ".csv"
  )
  df_list <- read_pseudobulk_expression(aggregated_count_file_list)

  # Drop failing samples for minimum cell count (10) and minimum total counts (1000) thresholds
  dropped <- subset(metadata_sub, keep == "False")
  drop_ids <- paste(dropped$sample.sampleKitGuid, dropped$aifi_plot_l3, sep = ":")
  df_list <- lapply(df_list, function(df) {
    df[, !(names(df) %in% drop_ids), drop = FALSE]
  })

  # Coverage check
  per_df_ct <- map(df_list, ~ {
    kit_id <- sub(":.*", "", names(.x)[1])
    tibble(
      kit_id   = kit_id,
      celltype = sub("^[^:]+:", "", names(.x))
    )
  })
  all_ct <- dplyr::bind_rows(per_df_ct)
  kit_info <- metadata_sub %>% distinct(sample.sampleKitGuid, label.visitDetails, subject.biologicalSex)

  all_ct_with_meta <- all_ct %>%
    left_join(kit_info, by = c("kit_id" = "sample.sampleKitGuid"))

  ct_coverage_by <- all_ct_with_meta %>%
    distinct(kit_id, celltype, label.visitDetails) %>%
    group_by(celltype, label.visitDetails) %>%
    summarise(n_kits = n(), .groups = "drop")

  # Celltype needs to be in at least 3 samples per condition to pass test
  n_of_three_cells <- unique(ct_coverage_by[ct_coverage_by$n_kits <= 3, ]$celltype)

  # Plot the cell type coverage across sex & time points
  plot_dir <- "../../../data/rna/pseudobulk/results/ct_coverage_plots"
  dir.create(plot_dir, recursive = TRUE, showWarnings = FALSE)

  p <- ggplot(ct_coverage_by, aes(x = n_kits, fill = label.visitDetails)) +
    geom_histogram(binwidth = 1, boundary = 0, color = "white", position = "dodge") +
    facet_wrap(~label.visitDetails, nrow = 1) +
    labs(
      title = paste0("Cell-type coverage: ", timepoint_1, " vs ", timepoint_2),
      x = "# of kits containing the cell type",
      y = "Number of cell types"
    ) +
    theme_bw() +
    theme(legend.position = "right")

  outfile_png <- file.path(
    plot_dir,
    paste0("ct_coverage_", gsub("[^A-Za-z0-9]+", "_", paste(timepoint_1, "vs", timepoint_2)), ".png")
  )
  ggsave(outfile_png, p, width = 7, height = 3, dpi = 300, bg = "white")

  all_combos <- expand_grid(
    celltype = unique(ct_coverage_by$celltype),
    label.visitDetails = unique(ct_coverage_by$label.visitDetails),
    subject.biologicalSex = unique(ct_coverage_by$subject.biologicalSex)
  )
  # also, drop any missing combinations
  missing_conditions <- all_combos %>%
    anti_join(ct_coverage_by, by = c("celltype", "label.visitDetails")) %>%
    pull(celltype) %>%
    unique()

  # Get unique cell types
  celltypes <- unique(unlist(lapply(df_list, names)))
  celltypes <- unique(sub(".*:", "", celltypes))
  celltypes <- setdiff(celltypes, c(n_of_three_cells, missing_conditions))

  # Load condition-specific gene list
  filtered_gene_set <- read.csv(file.path("../../../data/rna/pseudobulk/outputs", gene_file))

  # Subset metadata down to only relevant columns
  metadata_deseq2 <- metadata_sub %>%
    distinct(subject.subjectGuid, sample.sampleKitGuid, subject.biologicalSex, subject.age, label.visitDetails, cohort.cohortGuid, subject.cmv)

  # Run DESeq2
  failed_log <- list()
  res_list <- lapply(celltypes, function(celltype) {
    tryCatch(
      {
        # Subset columns for this celltype from each df
        celltype_list <- lapply(df_list, function(df) {
          df[, grep(celltype, names(df), fixed = TRUE), drop = FALSE]
        })

        # Combine and normalize sample IDs
        exp_matrix <- do.call(cbind, celltype_list)
        colnames(exp_matrix) <- sub(":.*", "", colnames(exp_matrix))

        if (any(duplicated(colnames(exp_matrix)))) {
          print(paste("Removing", sum(duplicated(colnames(exp_matrix))), "duplicate columns for celltype:", celltype))
          # Take only the first occurrence of each sample
          exp_matrix <- exp_matrix[, !duplicated(colnames(exp_matrix)), drop = FALSE]
        }

        # Align metadata rownames to sample IDs
        rownames(metadata_deseq2) <- metadata_deseq2$sample.sampleKitGuid

        # Gene list for this celltype
        filtered_genes <- filtered_gene_set %>%
          filter(aifi_plot_l3 == celltype) %>%
          pull(gene)

        # Run DESeq2
        res <- deseq2_analysis(
          exp_matrix,
          meta_data = metadata_deseq2,
          filtered_gene_set = filtered_genes,
          formula = ~ label.visitDetails + subject.biologicalSex + subject.age + subject.cmv,
          comparisons = list(c("label.visitDetails", timepoint_1, timepoint_2)),
          celltype = celltype
        )

        # Post-process
        res <- as.data.frame(res)
        res$Qvalue <- qvalue::qvalue(res$pvalue)$qvalues
        return(res)
      },
      error = function(e) {
        failed_log <<- append(failed_log, list(data.frame(
          timestamp = Sys.time(),
          celltype  = celltype,
          error     = as.character(e)
        )))
        return(NULL)
      }
    )
  })

  # Save failed cell types log if any failures occurred
  if (length(failed_log) > 0) {
    failed_df <- do.call(rbind, failed_log)
    failed_outfile <- paste0(
      "../../../data/rna/pseudobulk/results/deseq2_results/failed_celltypes_bmmc_",
      timepoint_1, "_vs_", timepoint_2, ".csv"
    )
    write.csv(failed_df, file = failed_outfile, row.names = FALSE)
  }

  # Combine results and write to file
  res <- do.call(rbind, res_list[!sapply(res_list, is.null)])
  write.csv(res, file = file.path("../../../data/rna/pseudobulk/results/deseq2_results", out_file), row.names = FALSE)
}

[1] "Total reading time: 5.77 seconds"
[1] "The length of the list matches the length of the input path."


Warning message:
“Unknown or uninitialised column: `subject.biologicalSex`.”
converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric variables that have mean or
  standard deviation larger than 5 (an arbitrary threshold to trigger this message).
  Including numeric variables with large mean can induce collinearity with the intercept.
  Users should center and scale numeric variables in the design to improve GLM convergence.

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not requir

[1] "Removing 18 duplicate columns for celltype: pDC"


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric variables that have mean or
  standard deviation larger than 5 (an arbitrary threshold to trigger this message).
  Including numeric variables with large mean can induce collinearity with the intercept.
  Users should center and scale numeric variables in the design to improve GLM convergence.

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are s

[1] "Removing 50 duplicate columns for celltype: Prog Ery"


...

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric variables that have mean or
  standard deviation larger than 5 (an arbitrary threshold to trigger this message).
  Including numeric variables with large mean can induce collinearity with the intercept.
  Users should center and scale numeric variables in the design to improve GLM convergence.

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are s

[1] "Removing 32 duplicate columns for celltype: Prog Ery"


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric variables that have mean or
  standard deviation larger than 5 (an arbitrary threshold to trigger this message).
  Including numeric variables with large mean can induce collinearity with the intercept.
  Users should center and scale numeric variables in the design to improve GLM convergence.

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are s

[1] "Removing 16 duplicate columns for celltype: Prog B"


converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric variables that have mean or
  standard deviation larger than 5 (an arbitrary threshold to trigger this message).
  Including numeric variables with large mean can induce collinearity with the intercept.
  Users should center and scale numeric variables in the design to improve GLM convergence.

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  the design formula contains one or mo